# MIRA | Treino do modelo de EPI (capacete + colete)

Esse notebook treina o modelo de visão computacional do MIRA na GPU grátis do Colab.

**Antes de tudo:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.

Depois é só ir em **Ambiente de execução → Executar tudo** e esperar (uns 20 a 30 min). No final o arquivo `mira_ppe.pt` é baixado automaticamente.

Não feche essa aba enquanto estiver rodando.

## 1. Confere se a GPU está ligada

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("Sem GPU! Vá em Ambiente de execução > Alterar o tipo > GPU T4 e rode de novo.")

## 2. Instala o Ultralytics (biblioteca do YOLO)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Baixa o dataset

Dataset **Construction Site Safety** (Roboflow, licença CC BY 4.0): 2.801 fotos de obra já anotadas.

In [ ]:
%cd /content
!rm -rf ds_repo
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/hoanganhtuoi125-gif/Construction-Site-Safety-PPE-Detection.git ds_repo
%cd ds_repo
!git sparse-checkout set Model-Training/Dataset
%cd /content
!for s in train valid test; do echo "$s: $(ls ds_repo/Model-Training/Dataset/$s/images | wc -l) imagens"; done

## 4. Filtra só o que o MVP precisa

O dataset original tem 10 classes (máquinas, cones, veículos...). Aqui ficamos só com 5:

| id | classe |
|---|---|
| 0 | capacete |
| 1 | sem_capacete |
| 2 | colete |
| 3 | sem_colete |
| 4 | pessoa |

In [ ]:
import os, glob, shutil
SRC = "/content/ds_repo/Model-Training/Dataset"
DST = "/content/dataset"
# ids originais: 0 Hardhat, 2 NO-Hardhat, 7 Safety Vest, 4 NO-Safety Vest, 5 Person
MAPA = {0: 0, 2: 1, 7: 2, 4: 3, 5: 4}
NOMES = ["capacete", "sem_capacete", "colete", "sem_colete", "pessoa"]

shutil.rmtree(DST, ignore_errors=True)
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DST}/{split}/images")
    os.makedirs(f"{DST}/{split}/labels")
    for img in glob.glob(f"{SRC}/{split}/images/*"):
        nome = os.path.splitext(os.path.basename(img))[0]
        shutil.copy(img, f"{DST}/{split}/images/")
        linhas = []
        arq = f"{SRC}/{split}/labels/{nome}.txt"
        if os.path.exists(arq):
            for linha in open(arq):
                p = linha.split()
                if not p or int(p[0]) not in MAPA:
                    continue
                v = list(map(float, p[1:]))
                if len(v) > 4:  # alguns rótulos vêm como polígono: vira caixa
                    xs, ys = v[0::2], v[1::2]
                    v = [(min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)]
                linhas.append(f"{MAPA[int(p[0])]} " + " ".join(f"{x:.6f}" for x in v[:4]))
        open(f"{DST}/{split}/labels/{nome}.txt", "w").write("\n".join(linhas) + "\n")

with open(f"{DST}/data.yaml", "w") as f:
    f.write(f"path: {DST}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n")
    for i, n in enumerate(NOMES):
        f.write(f"  {i}: {n}\n")
print(open(f"{DST}/data.yaml").read())

## 5. Treina o modelo

- Modelo base: **YOLO26n** (versão nano, leve o suficiente pra rodar em notebook sem GPU)
- 50 épocas: cada época é uma passada completa pelas fotos

Vai aparecer uma barra de progresso por época. As colunas que importam são **mAP50** (quanto mais perto de 1, melhor).

In [ ]:
from ultralytics import YOLO
modelo = YOLO("yolo26n.pt")
modelo.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    project="/content/runs",
    name="mira_ppe",
    exist_ok=True,
)

## 6. Resultado no conjunto de teste (fotos que o modelo nunca viu)

In [ ]:
modelo = YOLO("/content/runs/mira_ppe/weights/best.pt")
m = modelo.val(data="/content/dataset/data.yaml", split="test", imgsz=640)
print()
print(f"mAP50 geral: {m.box.map50:.2f}")
print("mAP50-95 por classe:")
for i, nome in modelo.names.items():
    print(f"  {nome:14s} {m.box.maps[i]:.2f}")

## 7. Vê o modelo funcionando em algumas fotos

In [ ]:
import glob, random
from IPython.display import Image, display
fotos = random.sample(glob.glob("/content/dataset/test/images/*"), 4)
res = modelo.predict(fotos, conf=0.35, imgsz=640, save=True, project="/content/runs", name="teste", exist_ok=True, verbose=False)
for f in glob.glob("/content/runs/teste/*.jpg")[:4]:
    display(Image(filename=f, width=500))

## 8. Baixa o modelo

Vai baixar o arquivo **mira_ppe.pt**. Coloca ele na mesma pasta do `mira_webcam.py` no seu notebook.

Se o download não começar sozinho, abre o ícone de pasta na barra lateral esquerda, acha `/content/mira_ppe.pt`, clica nos três pontinhos e em **Fazer download**.

In [ ]:
import shutil
shutil.copy("/content/runs/mira_ppe/weights/best.pt", "/content/mira_ppe.pt")
from google.colab import files
files.download("/content/mira_ppe.pt")